# 3.1 — Evapotranspiration accuracy assessment at the annual scale

The analysis compares annual CONUS404 actual evapotranspiration with every enabled reference product on the CONUS404 grid. The workflow treats each raster as an annual accumulated water depth in millimeters and uses pairwise finite, positive values for each model–reference comparison.

For modeled values $S_t$, reference values $O_t$, and annual index $t$, the analysis calculates:

$$
\mathrm{MBE}=\frac{1}{n}\sum_t(S_t-O_t),\qquad
\mathrm{MAE}=\frac{1}{n}\sum_t|S_t-O_t|
$$

$$
\mathrm{RMSE}=\sqrt{\frac{1}{n}\sum_t(S_t-O_t)^2},\qquad
\mathrm{PBIAS}(\%)=100\frac{\sum_t(S_t-O_t)}{\sum_t O_t}
$$

The analysis normalizes MAE by the multiannual mean of the corresponding modeled actual evapotranspiration, not by the reference mean:

$$
\mathrm{nMAE}(\%)=100\frac{\mathrm{MAE}}{\overline{ET}_{\mathrm{model}}}.
$$

Positive PBIAS represents model overestimation and negative PBIAS represents model underestimation.


## Configuration and input domain

The notebook reads the study period, enabled datasets, filename templates, climate-region layer, resampling method, figure ranges, and output directory from `config.yml`. The workflow uses inclusive start and end years.


In [ ]:
from pathlib import Path
import sys

import geopandas as gpd

REPOSITORY_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(REPOSITORY_ROOT / "src"))

from accuracy_assessment import (
    annual_pbias,
    compute_metrics,
    enabled_items,
    load_comparison,
    load_config,
    paired_valid,
    validate_inputs,
    write_metric_set,
    write_raster,
    years_for,
)
from regional_plots import plot_grouped_violins

COMPONENT = "evapotranspiration"
SCALE = "annual"
config = load_config(REPOSITORY_ROOT / "config.yml")
component_config = config["components"][COMPONENT]
models = enabled_items(component_config, "models")
references = enabled_items(component_config, "references")
regions = gpd.read_file(config["regions"]["file"])
years = years_for(config, SCALE, COMPONENT, assessment="accuracy")
output_dir = Path(config["output_dir"]) / "accuracy" / COMPONENT
output_dir.mkdir(parents=True, exist_ok=True)
print(f"{COMPONENT} {SCALE} period: {years[0]}–{years[-1]}")


## Input validation

The workflow requires one annual raster for every configured dataset and year and one climate-region polygon layer. It stops before calculation when an input is absent.


In [ ]:
validate_inputs(
    config,
    SCALE,
    years,
    component_config=component_config,
)


## Raster alignment and accuracy metrics

The workflow reprojects each reference-year raster to its corresponding CONUS404 grid with the configured resampling method. It applies the same validity mask to the modeled and reference arrays, calculates the complete metric set, and retains missing values as nodata. The analysis also calculates PBIAS independently for each year because the annual evapotranspiration PBIAS series is required by the component-wise correlation assessment.


In [ ]:
positive_only = config["processing"].get("require_positive_values", True)
resampling = config["processing"].get("resampling", "bilinear")
metric_paths = {}

for model_key, model in models.items():
    for reference_key, reference in references.items():
        comparison_key = f"{model_key}_vs_{reference_key}"
        model_raw, reference_raw = load_comparison(
            model,
            reference,
            SCALE,
            years,
            resampling=resampling,
        )
        model_paired, reference_paired = paired_valid(
            model_raw,
            reference_raw,
            positive_only=positive_only,
        )
        metrics = compute_metrics(
            model_paired,
            reference_paired,
            model_for_denominator=model_raw,
        )
        comparison_dir = output_dir / SCALE / comparison_key
        metric_paths[comparison_key] = write_metric_set(metrics, comparison_dir)
        if config["processing"].get("export_yearly_pbias", True):
            yearly_pbias = annual_pbias(model_paired, reference_paired)
            for year in years:
                write_raster(
                    yearly_pbias.sel(year=year),
                    comparison_dir / "annual_pbias" / f"wy{year}.tif",
                )
        print("Completed", comparison_key)


## Climate-region nMAE distributions

The workflow extracts every finite nMAE raster cell inside each climate region. The violin body represents the pixel distribution; the circle represents the median; the × represents the mean; and the error bar represents the mean ± one sample standard deviation. The workflow applies the configured display interval before calculating the displayed regional statistics, so the exported table describes the same values shown in the figure.


In [ ]:
region_config = config["regions"]
global_figure_config = config["figures"]
figure_config = global_figure_config["accuracy"][COMPONENT]
exclude_zero = config["processing"].get("exclude_exact_zero_from_distributions", False)
nmae_series = []
for model_key, model in models.items():
    for reference_key, reference in references.items():
        key = f"{model_key}_vs_{reference_key}"
        nmae_series.append(
            (f"{model['label']}\n{reference['label']}", metric_paths[key]["nmae_percent"])
        )
nmae_stats = plot_grouped_violins(
    nmae_series,
    regions,
    region_config["name_field"],
    region_config["code_field"],
    output_dir / "figures" / "annual_nmae_violin.png",
    output_dir / "tables" / "annual_nmae_regional_statistics.csv",
    "nMAE (%)",
    tuple(figure_config["annual_nmae_range"]),
    global_figure_config["colors"][:len(nmae_series)],
    dpi=global_figure_config["dpi"],
    exclude_zero=exclude_zero,
)
nmae_stats


## Climate-region PBIAS distributions

The analysis summarizes multiannual PBIAS with the same regional extraction and plotting procedure. The workflow retains exact zero values because zero is a valid unbiased result, and the workflow uses a horizontal zero line to separate positive and negative bias.


In [ ]:
pbias_series = []
for model_key, model in models.items():
    for reference_key, reference in references.items():
        key = f"{model_key}_vs_{reference_key}"
        pbias_series.append(
            (f"{model['label']}\n{reference['label']}", metric_paths[key]["pbias"])
        )
pbias_stats = plot_grouped_violins(
    pbias_series,
    regions,
    region_config["name_field"],
    region_config["code_field"],
    output_dir / "figures" / "annual_pbias_violin.png",
    output_dir / "tables" / "annual_pbias_regional_statistics.csv",
    "PBIAS (%)",
    tuple(figure_config["annual_pbias_range"]),
    global_figure_config["colors"][:len(pbias_series)],
    dpi=global_figure_config["dpi"],
    exclude_zero=False,
)
pbias_stats
